In [22]:
import pandas as pd
import numpy as np
import ast
import pandas_market_calendars as mcal

In [23]:
df_matched = pd.read_csv("tweets_matched.csv")
df_dict = pd.read_csv("company_dict_10_industries.csv")

## 1-Keep matched_tickers & generate new_id

In [24]:
# 1️⃣ Keep only tweets with non-empty matched_tickers
df_matched = df_matched[
    df_matched["matched_tickers"].notna() &
    (df_matched["matched_tickers"] != "[]")
].copy()

# 2️⃣ Reset index to ensure continuous numbering
df_matched = df_matched.reset_index(drop=True)

# 3️⃣ Create a new column: sequential number (starting from 1) 
new_id = (
    (df_matched.index + 1).astype(str) 
)

# 4️⃣ Insert as the first column
df_matched.insert(0, "new_id", new_id)

# Preview result
df_matched.head()



,new_id,id,text,isRetweet,isDeleted,device,favorites,retweets,date,isFlagged,datetime_raw,event_minute,matched_tickers,match_method,matched_aliases,matched_industries,is_quoted
0,1,1196121686728466434,Paul Krugman has called me wrong from day one....,f,f,Twitter for iPhone,32612,8641,2019-11-17 17:43:27,f,2019-11-17 17:43:27+00:00,2019-11-17 17:43:00+00:00,['NYT'],alias,['New York Times'],['News & Publishing'],False
1,2,1201639937268948992,Mini Mike Bloomberg has instructed his third r...,f,f,Twitter for iPhone,114699,28696,2019-12-02 23:11:01,f,2019-12-02 23:11:01+00:00,2019-12-02 23:11:00+00:00,['NYT'],alias,['New York Times'],['News & Publishing'],False
2,3,948202173049049088,The Failing New York Times has a new publisher...,f,f,Twitter for iPhone,66457,13909,2018-01-02 14:39:49,f,2018-01-02 14:39:49+00:00,2018-01-02 14:39:00+00:00,['NYT'],alias,['New York Times'],['News & Publishing'],False
3,4,812061677160202240,Based on the tremendous cost and cost overruns...,f,f,Twitter for Android,51313,11775,2016-12-22 22:26:05,f,2016-12-22 22:26:05+00:00,2016-12-22 22:26:00+00:00,"['LMT', 'BA']",name,"['Lockheed Martin', 'Boeing']","['Defense & Aerospace', 'Defense & Aerospace']",False
4,5,807932020236124160,I will be interviewed today on Fox News Sunday...,f,f,Twitter for Android,24897,4524,2016-12-11 12:56:18,f,2016-12-11 12:56:18+00:00,2016-12-11 12:56:00+00:00,['FOXA'],alias,['Fox News'],['News & Publishing'],False


## 2- One row per ticker/company/industry

In [25]:
# Convert matched_tickers from string to list
df_matched["matched_tickers"] = df_matched["matched_tickers"].apply(ast.literal_eval)

# Explode into one row per ticker
df_exploded = df_matched.explode("matched_tickers")

# Rename column
df_exploded = df_exploded.rename(columns={"matched_tickers": "ticker"})

# Merge with dictionary to get company and industry
df_exploded = df_exploded.merge(
    df_dict[["ticker", "company_name", "industry"]],
    on="ticker",
    how="left"
)

# Check result
df_exploded.head()

,new_id,id,text,isRetweet,isDeleted,device,favorites,retweets,date,isFlagged,datetime_raw,event_minute,ticker,match_method,matched_aliases,matched_industries,is_quoted,company_name,industry
0,1,1196121686728466434,Paul Krugman has called me wrong from day one....,f,f,Twitter for iPhone,32612,8641,2019-11-17 17:43:27,f,2019-11-17 17:43:27+00:00,2019-11-17 17:43:00+00:00,NYT,alias,['New York Times'],['News & Publishing'],False,The New York Times Company,News & Publishing
1,2,1201639937268948992,Mini Mike Bloomberg has instructed his third r...,f,f,Twitter for iPhone,114699,28696,2019-12-02 23:11:01,f,2019-12-02 23:11:01+00:00,2019-12-02 23:11:00+00:00,NYT,alias,['New York Times'],['News & Publishing'],False,The New York Times Company,News & Publishing
2,3,948202173049049088,The Failing New York Times has a new publisher...,f,f,Twitter for iPhone,66457,13909,2018-01-02 14:39:49,f,2018-01-02 14:39:49+00:00,2018-01-02 14:39:00+00:00,NYT,alias,['New York Times'],['News & Publishing'],False,The New York Times Company,News & Publishing
3,4,812061677160202240,Based on the tremendous cost and cost overruns...,f,f,Twitter for Android,51313,11775,2016-12-22 22:26:05,f,2016-12-22 22:26:05+00:00,2016-12-22 22:26:00+00:00,LMT,name,"['Lockheed Martin', 'Boeing']","['Defense & Aerospace', 'Defense & Aerospace']",False,Lockheed Martin,Defense & Aerospace
4,4,812061677160202240,Based on the tremendous cost and cost overruns...,f,f,Twitter for Android,51313,11775,2016-12-22 22:26:05,f,2016-12-22 22:26:05+00:00,2016-12-22 22:26:00+00:00,BA,name,"['Lockheed Martin', 'Boeing']","['Defense & Aerospace', 'Defense & Aerospace']",False,Boeing Company,Defense & Aerospace


## 3-Keep only first tweet in each 1-hour cluster

In [26]:
# Ensure datetime format (UTC)
df_exploded["event_minute"] = pd.to_datetime(df_exploded["event_minute"], utc=True)

# Sort by ticker and time
df_exploded = df_exploded.sort_values(["ticker", "event_minute"])

# Compute time difference within same ticker
df_exploded["time_diff"] = (
    df_exploded.groupby("ticker")["event_minute"]
    .diff()
)

# Flag tweets within 1 hour
df_exploded["within_1h"] = df_exploded["time_diff"] <= pd.Timedelta(hours=1)

# 🔎 Print all tweets that are within 1 hour of previous tweet (same ticker)
result = df_exploded[df_exploded["within_1h"]]

print("Total tweets within 1 hour (same ticker):", len(result))
print(result[["new_id", "text","ticker" ,"event_minute", "time_diff"]])

Total tweets within 1 hour (same ticker): 24
    new_id                                               text ticker  \
24      23  Boycott all Apple products  until such time as...   AAPL   
59      58  So many stories about me in the @washingtonpos...   AMZN   
58      57  Is Fake News Washington Post being used as a l...   AMZN   
136    131  ...does not include the Fake Washington Post, ...   AMZN   
115    111  ....In my opinion the Washington Post is nothi...   AMZN   
47      46  Broadcom's move to America=$20 BILLION of annu...   AVGO   
43      42  My thoughts and prayers are with everyone invo...     DD   
81      78  Very disappointed with General Motors and thei...     GM   
99      95  ....results on “Trump News” are from National ...  GOOGL   
98      94  Google search results for “Trump News” shows o...  GOOGL   
245    237  Just met with @SundarPichai, President of @Goo...  GOOGL   
206    199  ....are NOT planning to illegally subvert the ...  GOOGL   
207    200  @sundar

In [27]:
# Ensure datetime format (UTC)
df_exploded["event_minute"] = pd.to_datetime(df_exploded["event_minute"], utc=True)

# Sort by ticker and time
df_exploded = df_exploded.sort_values(["ticker", "event_minute"])

# Compute time difference within same ticker
df_exploded["time_diff"] = (
    df_exploded.groupby("ticker")["event_minute"]
    .diff()
)

# Keep only first tweet in each 1-hour cluster
df_cleaned = df_exploded[
    (df_exploded["time_diff"].isna()) | 
    (df_exploded["time_diff"] > pd.Timedelta(hours=1))
].copy()

print("Original rows:", len(df_exploded))
print("After removing 1-hour duplicates:", len(df_cleaned))

Original rows: 262
After removing 1-hour duplicates: 238


## 4- In trading hours

In [28]:
#whether the tweet date is a valid NYSE trading day (excluding weekends and market holidays) and then verifies if the timestamp falls within official market hours (09:30–16:00 ET, including early close days). If either condition is not satisfied, the observation is classified as outside trading hours (in_trading_hours = 1); otherwise, it is marked as during trading hours (0)

# Ensure UTC timestamp
df_cleaned["event_minute"] = pd.to_datetime(
    df_cleaned["event_minute"], 
    utc=True,
    errors="coerce"
)

# Convert to US Eastern Time
df_cleaned["event_time_et"] = df_cleaned["event_minute"].dt.tz_convert("America/New_York")

# Get NYSE trading schedule covering event range
nyse = mcal.get_calendar("NYSE")

start_date = df_cleaned["event_time_et"].min().strftime("%Y-%m-%d")
end_date   = df_cleaned["event_time_et"].max().strftime("%Y-%m-%d")

schedule = nyse.schedule(start_date=start_date, end_date=end_date)

# Create a lookup set of valid trading days
trading_days = set(schedule.index.date)

# Function to check trading hours
def is_outside_trading(row):
    if pd.isna(row["event_time_et"]):
        return 1  # treat missing as outside
    
    event_dt = row["event_time_et"]
    event_date = event_dt.date()

    # Not a trading day (weekend or holiday)
    if event_date not in trading_days:
        return 1

    # Get market open/close for that date
    market_open = schedule.loc[str(event_date)]["market_open"]
    market_close = schedule.loc[str(event_date)]["market_close"]

    # Outside intraday session
    if event_dt < market_open or event_dt > market_close:
        return 1

    return 0  # inside trading hours

df_cleaned["in_trading_hours"] = df_cleaned.apply(is_outside_trading, axis=1)

# Quick summary
print(df_cleaned["in_trading_hours"].value_counts())

in_trading_hours
1    184
0     54
Name: count, dtype: int64


## 5- Transfer time

In [29]:
# ============================================================
# 1️⃣ Ensure event_time_et is proper timezone-aware ET datetime
# ============================================================

df_cleaned["event_time_et"] = pd.to_datetime(
    df_cleaned["event_time_et"],
    errors="coerce"
)

if df_cleaned["event_time_et"].dt.tz is None:
    df_cleaned["event_time_et"] = df_cleaned["event_time_et"].dt.tz_localize(
        "America/New_York"
    )

# ============================================================
# 2️⃣ Build NYSE trading schedule (add buffer to avoid overflow)
# ============================================================

nyse = mcal.get_calendar("NYSE")

start_date = df_cleaned["event_time_et"].min().strftime("%Y-%m-%d")
end_date = (
    df_cleaned["event_time_et"].max() + pd.Timedelta(days=10)
).strftime("%Y-%m-%d")

schedule = nyse.schedule(start_date=start_date, end_date=end_date)

# Force schedule to ET (important)
schedule["market_open"] = schedule["market_open"].dt.tz_convert("America/New_York")
schedule["market_close"] = schedule["market_close"].dt.tz_convert("America/New_York")

# ============================================================
# 3️⃣ Map each event to market open/close of that day
# ============================================================

schedule_map = schedule.copy()
schedule_map["date"] = schedule_map.index.date
schedule_map = schedule_map.set_index("date")

event_dates = df_cleaned["event_time_et"].dt.date

df_cleaned["market_open"] = event_dates.map(schedule_map["market_open"])
df_cleaned["market_close"] = event_dates.map(schedule_map["market_close"])

# ============================================================
# 4️⃣ Identify events inside trading hours
# ============================================================

is_trading_day = event_dates.isin(schedule_map.index)

inside_hours = (
    is_trading_day &
    (df_cleaned["event_time_et"] >= df_cleaned["market_open"]) &
    (df_cleaned["event_time_et"] <= df_cleaned["market_close"])
)

# ============================================================
# 5️⃣ Compute next trading day open (vectorized, safe)
# ============================================================

# Trading dates only (tz-naive)
trading_dates = pd.to_datetime(schedule.index.date)

event_dates_ts = pd.to_datetime(event_dates)

next_idx = trading_dates.searchsorted(event_dates_ts, side="right")

# Guard overflow
next_idx = np.clip(next_idx, 0, len(schedule) - 1)

next_open_series = schedule["market_open"].iloc[next_idx]
next_open_series = next_open_series.reset_index(drop=True)
next_open_series.index = df_cleaned.index

# ============================================================
# 6️⃣ Adjust event time
# ============================================================

df_cleaned["adjusted_event_time_et"] = df_cleaned["event_time_et"]

df_cleaned.loc[~inside_hours, "adjusted_event_time_et"] = (
    next_open_series[~inside_hours]
)

# ============================================================
# 7️⃣ Convert back to UTC
# ============================================================

df_cleaned["adjusted_event_time_utc"] = (
    df_cleaned["adjusted_event_time_et"]
    .dt.tz_convert("UTC")
)

df_cleaned.drop(columns=["market_open", "market_close"], inplace=True)

print("Done.")
print(df_cleaned["adjusted_event_time_et"].dtype)
print(df_cleaned["adjusted_event_time_utc"].dtype)

Done.
datetime64[ns, America/New_York]
datetime64[ns, UTC]


## 6-Event window

In [30]:


# =====================================================
# 1️⃣ Ensure adjusted_event_time_et is tz-aware ET
# =====================================================

df_cleaned["adjusted_event_time_et"] = pd.to_datetime(
    df_cleaned["adjusted_event_time_et"],
    errors="coerce"
)

if df_cleaned["adjusted_event_time_et"].dt.tz is None:
    df_cleaned["adjusted_event_time_et"] = (
        df_cleaned["adjusted_event_time_et"]
        .dt.tz_localize("America/New_York")
    )

# =====================================================
# 2️⃣ 1-hour window (ET first)
# =====================================================

df_cleaned["event_1h_pre_et"] = (
    df_cleaned["adjusted_event_time_et"] - pd.Timedelta(hours=1)
)

df_cleaned["event_1h_end_et"] = (
    df_cleaned["adjusted_event_time_et"] + pd.Timedelta(hours=1)
)

# =====================================================
# 3️⃣ 5-day trading window (ET based)
# =====================================================

nyse = mcal.get_calendar("NYSE")

start_date = df_cleaned["adjusted_event_time_et"].min().strftime("%Y-%m-%d")
end_date = (
    df_cleaned["adjusted_event_time_et"].max() + pd.Timedelta(days=20)
).strftime("%Y-%m-%d")

schedule = nyse.schedule(start_date=start_date, end_date=end_date)

# convert schedule to ET
schedule["market_open"] = schedule["market_open"].dt.tz_convert("America/New_York")

trading_dates = pd.to_datetime(schedule.index.date)

event_dates = pd.to_datetime(
    df_cleaned["adjusted_event_time_et"].dt.date
)

event_idx = trading_dates.searchsorted(event_dates)
event_idx = np.clip(event_idx, 0, len(trading_dates) - 1)

pre_idx = np.clip(event_idx - 5, 0, len(trading_dates) - 1)
post_idx = np.clip(event_idx + 5, 0, len(trading_dates) - 1)

# IMPORTANT: keep as Series (no .values)
event_5d_pre_series = schedule["market_open"].iloc[pre_idx]
event_5d_pre_series = event_5d_pre_series.reset_index(drop=True)
event_5d_pre_series.index = df_cleaned.index

event_5d_end_series = schedule["market_open"].iloc[post_idx]
event_5d_end_series = event_5d_end_series.reset_index(drop=True)
event_5d_end_series.index = df_cleaned.index

df_cleaned["event_5d_pre_et"] = event_5d_pre_series
df_cleaned["event_5d_end_et"] = event_5d_end_series

# =====================================================
# 4️⃣ Convert everything to UTC at the very end
# =====================================================

df_cleaned["event_1h_pre"] = df_cleaned["event_1h_pre_et"].dt.tz_convert("UTC")
df_cleaned["event_1h_end"] = df_cleaned["event_1h_end_et"].dt.tz_convert("UTC")

df_cleaned["event_5d_pre"] = df_cleaned["event_5d_pre_et"].dt.tz_convert("UTC")
df_cleaned["event_5d_end"] = df_cleaned["event_5d_end_et"].dt.tz_convert("UTC")


## 7-Drop unneccessary columns

In [31]:
print(df_cleaned.columns.tolist())

['new_id', 'id', 'text', 'isRetweet', 'isDeleted', 'device', 'favorites', 'retweets', 'date', 'isFlagged', 'datetime_raw', 'event_minute', 'ticker', 'match_method', 'matched_aliases', 'matched_industries', 'is_quoted', 'company_name', 'industry', 'time_diff', 'within_1h', 'event_time_et', 'in_trading_hours', 'adjusted_event_time_et', 'adjusted_event_time_utc', 'event_1h_pre_et', 'event_1h_end_et', 'event_5d_pre_et', 'event_5d_end_et', 'event_1h_pre', 'event_1h_end', 'event_5d_pre', 'event_5d_end']


In [32]:
columns_to_drop = [
    "isRetweet",
    "isDeleted",
    "device",
    "favorites",
    "retweets",
    "isFlagged",
    "datetime_raw",
    "match_method",
    "matched_aliases",
    "matched_industries",
    "is_quoted",
    "time_diff",
    "within_1h",
]

df_cleaned.drop(columns=columns_to_drop, inplace=True, errors="ignore")

In [33]:
df_cleaned["new_id"].nunique()

228

In [34]:
#sample.to_csv("tweets_random.csv", index=False)
df_cleaned.to_csv("tweets_event_windows_final.csv", index=False)